In [1]:
# =====================================================================
# HỆ THỐNG BASELINE: TF-IDF + LOGISTIC REGRESSION & SVM (KAGGLE VERSION)
# =====================================================================
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("🔍 Đang rà quét hệ thống tìm dữ liệu...")
train_path = dev_path = test_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train_clean_PhoBERT.csv' in files: train_path = os.path.join(root, 'train_clean_PhoBERT.csv')
    if 'dev_clean_PhoBERT.csv' in files: dev_path = os.path.join(root, 'dev_clean_PhoBERT.csv')
    if 'test_clean_PhoBERT.csv' in files: test_path = os.path.join(root, 'test_clean_PhoBERT.csv')

if not all([train_path, dev_path, test_path]):
    raise FileNotFoundError("❌ Thiếu file! Hãy bấm 'Add Data' để nạp Dataset vào Notebook.")

print(f"✅ Đã tìm thấy dữ liệu! Bắt đầu nạp...\n{train_path}")

# 1. Nạp và xử lý NaN
train_df = pd.read_csv(train_path).dropna(subset=['clean_text_PhoBERT'])
val_df = pd.read_csv(dev_path).dropna(subset=['clean_text_PhoBERT'])
test_df = pd.read_csv(test_path).dropna(subset=['clean_text_PhoBERT'])

X_train, X_val, X_test = train_df['clean_text_PhoBERT'], val_df['clean_text_PhoBERT'], test_df['clean_text_PhoBERT']
y_train_sentiment, y_train_topic = train_df['sentiment'], train_df['topic']
y_val_sentiment, y_val_topic = val_df['sentiment'], val_df['topic']
y_test_sentiment, y_test_topic = test_df['sentiment'], test_df['topic']

# 2. Rút trích đặc trưng TF-IDF
print("\n⏳ Đang chuyển hóa văn bản bằng TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 3. Huấn luyện các mô hình Baseline (Tuần tự)
print("🚀 Đang huấn luyện Logistic Regression và SVM...")
models = {
    'lr': {'sentiment': LogisticRegression(max_iter=1000, class_weight='balanced'), 
           'topic': LogisticRegression(max_iter=1000, class_weight='balanced')},
    'svm': {'sentiment': SVC(kernel='linear', class_weight='balanced'), 
            'topic': SVC(kernel='linear', class_weight='balanced')}
}

predictions = {'lr': {}, 'svm': {}}

for model_name, model_dict in models.items():
    print(f"   + Huấn luyện {model_name.upper()}...")
    model_dict['sentiment'].fit(X_train_tfidf, y_train_sentiment)
    model_dict['topic'].fit(X_train_tfidf, y_train_topic)
    
    # Lưu dự đoán vào Dictionary
    predictions[model_name]['sentiment'] = model_dict['sentiment'].predict(X_test_tfidf)
    predictions[model_name]['topic'] = model_dict['topic'].predict(X_test_tfidf)

print("✅ Huấn luyện hoàn tất!\n")

# =====================================================================
# XUẤT KẾT QUẢ RA FILE (ARTIFACTS) ĐỂ LÀM BÁO CÁO
# =====================================================================
out_dir = "/kaggle/working/"
sentiment_labels = ['Tiêu cực (0)', 'Trung tính (1)', 'Tích cực (2)']
topic_labels = ['Giảng viên (0)', 'Chương trình (1)', 'Cơ sở vật chất (2)', 'Khác (3)']

print("📊 Đang xuất báo cáo số liệu và vẽ ma trận nhầm lẫn...")

for model_name, pred_dict in predictions.items():
    # A. XUẤT CSV PER-CLASS REPORT
    sent_dict = classification_report(y_test_sentiment, pred_dict['sentiment'], target_names=sentiment_labels, output_dict=True)
    pd.DataFrame(sent_dict).transpose().round(4).to_csv(os.path.join(out_dir, f"baseline_{model_name}_sentiment_report.csv"), encoding='utf-8-sig')
    
    top_dict = classification_report(y_test_topic, pred_dict['topic'], target_names=topic_labels, output_dict=True)
    pd.DataFrame(top_dict).transpose().round(4).to_csv(os.path.join(out_dir, f"baseline_{model_name}_topic_report.csv"), encoding='utf-8-sig')

    # B. VẼ MA TRẬN NHẦM LẪN (CONFUSION MATRIX)
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.heatmap(confusion_matrix(y_test_topic, pred_dict['topic']), annot=True, fmt='d', cmap='Blues', ax=ax[0], xticklabels=topic_labels, yticklabels=topic_labels)
    ax[0].set_title(f'Baseline {model_name.upper()}: Chủ đề (Topic)')
    ax[0].set_xlabel('Dự đoán')
    ax[0].set_ylabel('Thực tế')
    
    sns.heatmap(confusion_matrix(y_test_sentiment, pred_dict['sentiment']), annot=True, fmt='d', cmap='Greens', ax=ax[1], xticklabels=sentiment_labels, yticklabels=sentiment_labels)
    ax[1].set_title(f'Baseline {model_name.upper()}: Cảm xúc (Sentiment)')
    ax[1].set_xlabel('Dự đoán')
    ax[1].set_ylabel('Thực tế')
    
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f"baseline_{model_name}_cm.png"))
    plt.close(fig) # Đóng để tránh in đè lên nhau

# C. XUẤT TOÀN BỘ CÂU SAI ĐỂ LÀM ERROR ANALYSIS CHO BASELINE SVM (Mô hình tốt nhất)
df_all_errors = pd.DataFrame({
    'clean_text_PhoBERT': X_test.values,
    'true_sentiment': y_test_sentiment.values,
    'pred_sentiment': predictions['svm']['sentiment'],
    'true_topic': y_test_topic.values,
    'pred_topic': predictions['svm']['topic']
})

# Lọc câu sai ở bất kỳ nhánh nào
mask_error = (df_all_errors['true_sentiment'] != df_all_errors['pred_sentiment']) | (df_all_errors['true_topic'] != df_all_errors['pred_topic'])
df_all_errors = df_all_errors[mask_error].copy()

# Ánh xạ nhãn và xuất file
df_all_errors['true_sentiment'] = df_all_errors['true_sentiment'].map({0: 'Tiêu cực', 1: 'Trung tính', 2: 'Tích cực'})
df_all_errors['pred_sentiment'] = df_all_errors['pred_sentiment'].map({0: 'Tiêu cực', 1: 'Trung tính', 2: 'Tích cực'})
df_all_errors['true_topic'] = df_all_errors['true_topic'].map({0: 'Giảng viên', 1: 'Chương trình', 2: 'Cơ sở vật chất', 3: 'Khác'})
df_all_errors['pred_topic'] = df_all_errors['pred_topic'].map({0: 'Giảng viên', 1: 'Chương trình', 2: 'Cơ sở vật chất', 3: 'Khác'})
df_all_errors.to_csv(os.path.join(out_dir, "baseline_svm_all_errors.csv"), index=False, encoding='utf-8-sig')

print(f"🎉 Hoàn tất toàn bộ quy trình! Các file kết quả đã được lưu tại Output (/kaggle/working/).")

🔍 Đang rà quét hệ thống tìm dữ liệu...
✅ Đã tìm thấy dữ liệu! Bắt đầu nạp...
/kaggle/input/datasets/conbobietbay/phobert-student-feedback-clean/train_clean_PhoBERT.csv

⏳ Đang chuyển hóa văn bản bằng TF-IDF...
🚀 Đang huấn luyện Logistic Regression và SVM...
   + Huấn luyện LR...
   + Huấn luyện SVM...
✅ Huấn luyện hoàn tất!

📊 Đang xuất báo cáo số liệu và vẽ ma trận nhầm lẫn...
🎉 Hoàn tất toàn bộ quy trình! Các file kết quả đã được lưu tại Output (/kaggle/working/).
